# Multi-Horizon Replication -- 7-Day and 30-Day Direction (Leakage-Safe, Original Data)

Forked from `original_model_replication.ipynb` with one change: the target is not tomorrow's
direction, it's whether BTC is bullish or bearish **7 calendar days out** and, separately,
**30 calendar days out**. Everything else is identical to that notebook -- the same 14
hand-picked features (`bitcoin-price.csv`, `market-data.csv`, `on-chain-data.csv`,
`economic-more.csv`), the same four models (Logistic Regression, Random Forest, LSTM,
CNN-LSTM), the same 8-fold expanding walk-forward, and the same McNemar's test +
fold-clustered bootstrap significance testing against an Always-up baseline.

**Why two separate horizons, not one:** they use the same feature set and evaluation
machinery but produce two independent target columns, models, and significance tables --
7-day and 30-day results are not directly comparable to each other (different base rates,
different effective sample sizes) any more than either is comparable to the original
1-day notebook.

**A caveat longer horizons introduce that 1-day-ahead didn't have:** multi-day targets
overlap. Two adjacent test rows' 30-day targets share up to 29 of their 30 look-ahead days,
so daily rows within a fold are far from independent -- unlike the 1-day notebook, where each
row's target barely overlaps its neighbor's. The fold-clustered bootstrap here already
treats each fold as a single resampling unit rather than each day, which is the right way to
avoid manufacturing false precision from that within-fold correlation, but it means the
existing "only 8 clusters" caveat applies with extra force: overlapping windows shrink the
effective number of independent observations per fold well below what the daily row count
suggests. Read every CI here as more uncertain than the same-width CI in the 1-day
notebook.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from scipy.stats import binomtest
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
np.random.seed(42)
pd.set_option("display.max_columns", 50)

ORIG_DATA_DIR = Path("../data/original")
MK4_DIR = Path("../data/mk4")
print("Imports OK. Torch:", torch.__version__)


Imports OK. Torch: 2.12.1


## 1. Load and Merge All Data Sources

Identical to `original_model_replication.ipynb` §1: loads the original notebook's own CSVs
directly and inner-joins every source on `date`, so the merged table is confined to the
single overlapping timeframe all four sources actually share.

In [2]:
btc_raw = pd.read_csv(ORIG_DATA_DIR / "bitcoin-price.csv")
market_raw = pd.read_csv(ORIG_DATA_DIR / "market-data.csv")
onchain_raw = pd.read_csv(ORIG_DATA_DIR / "on-chain-data.csv")
econ_more_raw = pd.read_csv(ORIG_DATA_DIR / "economic-more.csv")

btc_raw["date"] = pd.to_datetime(btc_raw["Datetime"]).dt.tz_localize(None).dt.normalize()
market_raw["date"] = pd.to_datetime(market_raw["Dates"])
onchain_raw["date"] = pd.to_datetime(onchain_raw["Datetime"]).dt.tz_localize(None).dt.normalize()
econ_more_raw["date"] = pd.to_datetime(econ_more_raw["Dates"])

btc = (btc_raw[["date", "Close"]]
       .rename(columns={"Close": "btc_close"})
       .dropna(subset=["btc_close"]))

market = market_raw.rename(columns={
    "SPX Index": "SP500", "CCMP Index": "Nasdaq", "INDU Index": "Dow",
    "XAU Curncy": "Gold", "XAG Curncy": "Silver", "LMCADY Comdty": "Copper",
})[["date", "SP500", "Nasdaq", "Dow", "Gold", "Silver", "Copper"]]

onchain = onchain_raw.rename(columns={
    "Miner Reserve": "miner_reserve", "Hashrate": "hashrate",
    "Mean Coin Age": "mean_coin_age", "Open Interest": "open_interest",
})[["date", "miner_reserve", "hashrate", "mean_coin_age", "open_interest"]]

econ = econ_more_raw.rename(columns={
    "FDTR Index": "Fed Funds Rate", "USGG10YR Index": "10Y Treasury Yield",
})[["date", "Fed Funds Rate", "10Y Treasury Yield"]]

# Inner-join every source on date (matching the original notebook's own merge strategy):
# every feature is confined to the exact same overlapping timeframe, no ffill across
# mismatched calendars (market-data.csv is weekdays-only, on-chain-data.csv is every day).
raw = (btc.merge(market, on="date", how="inner")
          .merge(onchain, on="date", how="inner")
          .merge(econ, on="date", how="inner"))
raw = raw.dropna().sort_values("date").reset_index(drop=True)

raw_start = raw["date"].min().date()
raw_end = raw["date"].max().date()
print(f"Merged raw table: {raw.shape}, {raw_start} to {raw_end}")
raw.tail()


Merged raw table: (1616, 14), 2019-04-01 to 2025-06-09


,date,btc_close,SP500,Nasdaq,Dow,Gold,Silver,Copper,miner_reserve,hashrate,mean_coin_age,open_interest,Fed Funds Rate,10Y Treasury Yield
1611,2025-06-03,105391.152971,5970.37,19398.96,42519.64,3353.43,34.5176,9686.31,1.808612e+06,9.089746e+11,1614.212233,3.547176e+10,4.5,4.4537
1612,2025-06-04,104724.591994,5970.81,19460.49,42427.74,3372.72,34.5045,9669.98,1.807733e+06,7.953528e+11,1613.870778,3.474156e+10,4.5,4.3552
1613,2025-06-05,101613.260806,5939.30,19298.45,42319.74,3352.65,35.6510,9832.65,1.807730e+06,8.458514e+11,1614.501350,3.385101e+10,4.5,4.3906
1614,2025-06-06,104406.924809,6000.36,19529.95,42762.87,3310.42,35.9777,9762.84,1.807638e+06,8.458514e+11,1615.110946,3.543465e+10,4.5,4.5056
1615,2025-06-09,110303.871227,6005.88,19591.24,42761.76,3326.19,36.7584,9888.78,1.808101e+06,8.837253e+11,1617.120227,3.784251e+10,4.5,4.4738


## 2. Feature Transforms

Identical to `original_model_replication.ipynb` §2: replicates the original notebook's exact
transform choices for all 14 of its final hand-picked features.

In [3]:
out = raw.sort_values("date").reset_index(drop=True).copy()

# Market (replicated exactly: simple pct change, matching "Tr S&P500" etc.)
out["Tr SP500"] = out["SP500"].pct_change()
out["Tr Nasdaq"] = out["Nasdaq"].pct_change()
out["Tr Dow"] = out["Dow"].pct_change()
out["Tr Gold"] = out["Gold"].pct_change()
out["Tr Silver"] = out["Silver"].pct_change()
out["Tr Copper"] = out["Copper"].pct_change()

# Economic (replicated exactly)
out["Tr Fed Funds Raw"] = out["Fed Funds Rate"]
out["Tr Fed Funds Delta"] = out["Fed Funds Rate"] - out["Fed Funds Rate"].shift(1)
out["Tr Treasury Pct"] = out["10Y Treasury Yield"].pct_change()
out["Tr Treasury Delta"] = out["10Y Treasury Yield"] - out["10Y Treasury Yield"].shift(5)

# On-chain (replicated exactly, matching model_df's "Tr Hashrate 1", "Tr Miner Reserve 1",
# "Tr Mean Coin Age 2", "Tr Open Interest" in the original notebook)
out["Tr Hashrate"] = out["hashrate"].rolling(7).mean()
out["Tr Miner Reserve"] = out["miner_reserve"] - out["miner_reserve"].shift(7)
out["Tr Mean Coin Age"] = out["mean_coin_age"].pct_change(7)
out["Tr Open Interest"] = out["open_interest"].pct_change()

FEATURE_COLS = [c for c in out.columns if c.startswith("Tr ")]
print(f"Replicated features ({len(FEATURE_COLS)}):")
for f in FEATURE_COLS:
    print(" ", f)


Replicated features (14):
  Tr SP500
  Tr Nasdaq
  Tr Dow
  Tr Gold
  Tr Silver
  Tr Copper
  Tr Fed Funds Raw
  Tr Fed Funds Delta
  Tr Treasury Pct
  Tr Treasury Delta
  Tr Hashrate
  Tr Miner Reserve
  Tr Mean Coin Age
  Tr Open Interest


## 3. Targets and Chronological Frame

Two targets instead of one: `target_bullish_7d` and `target_bullish_30d`, each defined the
same way as the original's 1-day target (`close.shift(-h) > close`) just with `h` = 7 or 30.
Both are built from the same feature-and-date frame so they share an identical set of usable
rows -- the last 30 rows (the longer horizon) are dropped since that target is unknowable,
which also trims the shorter horizon's frame by the same amount for a fair side-by-side
read.

In [4]:
HORIZON_TARGETS = {7: "target_bullish_7d", 30: "target_bullish_30d"}
MAX_HORIZON = max(HORIZON_TARGETS)

for h, col in HORIZON_TARGETS.items():
    out[col] = (out["btc_close"].shift(-h) > out["btc_close"]).astype("float")

out = out.iloc[:-MAX_HORIZON]  # last MAX_HORIZON rows' longest-horizon target is unknowable

model_frame = out[["date", "btc_close"] + list(HORIZON_TARGETS.values()) + FEATURE_COLS].copy()
model_frame = model_frame.replace([np.inf, -np.inf], np.nan)
model_frame = model_frame.dropna().sort_values("date").reset_index(drop=True)
for col in HORIZON_TARGETS.values():
    model_frame[col] = model_frame[col].astype(int)

model_frame.to_csv(MK4_DIR / "multiday_replication_model_frame.csv", index=False)
model_start = model_frame["date"].min().date()
model_end = model_frame["date"].max().date()
print(f"Model frame: {model_frame.shape}, {model_start} to {model_end}")
for h, col in HORIZON_TARGETS.items():
    print(f"  {h}-day up-rate: {model_frame[col].mean():.4f}")
model_frame.tail()


Model frame: (1579, 18), 2019-04-10 to 2025-04-28
  7-day up-rate: 0.5421
  30-day up-rate: 0.5668


,date,btc_close,target_bullish_7d,target_bullish_30d,Tr SP500,Tr Nasdaq,Tr Dow,Tr Gold,Tr Silver,Tr Copper,Tr Fed Funds Raw,Tr Fed Funds Delta,Tr Treasury Pct,Tr Treasury Delta,Tr Hashrate,Tr Miner Reserve,Tr Mean Coin Age,Tr Open Interest
1574,2025-04-22,93480.043611,1,1,0.025117,0.027063,0.026632,-0.012661,-0.005698,0.020904,4.5,0.0,-0.002222,0.0678,8.758208e+11,263.897444,0.003380,0.126477
1575,2025-04-23,93715.481248,1,1,0.016661,0.025007,0.010707,-0.027300,0.032876,0.001167,4.5,0.0,-0.004454,0.1044,8.470176e+11,175.035997,0.002321,-0.046458
1576,2025-04-24,93986.136921,1,1,0.020259,0.027411,0.012292,0.018578,0.000048,0.003797,4.5,0.0,-0.015133,-0.0100,8.461578e+11,-237.961268,0.002104,0.015710
1577,2025-04-25,94707.871375,1,1,0.007373,0.012635,0.000501,-0.008870,-0.013993,-0.002396,4.5,0.0,-0.018448,-0.0896,8.444719e+11,-400.010794,0.001518,0.038529
1578,2025-04-28,95011.413572,1,1,0.000641,-0.000967,0.002844,0.007308,0.001598,0.003220,4.5,0.0,-0.006399,-0.2024,8.358462e+11,-1187.014340,0.002570,-0.057704


## 4. Models

Identical to the original: Logistic Regression and Random Forest (sklearn), plus LSTM and
CNN-LSTM (PyTorch). The two sequence models still take a trailing 10-day window of the same
daily feature vector, regardless of which horizon's target they're being trained against --
only the label changes between horizons, not the feature history feeding the model.

In [5]:
SEQ_LOOKBACK = 10


def make_sequences(X, y, dates, lookback):
    Xs, ys, ds = [], [], []
    for i in range(lookback, len(X)):
        Xs.append(X[i - lookback:i])
        ys.append(y[i])
        ds.append(dates[i])
    return np.array(Xs), np.array(ys), np.array(ds)


class LSTMClassifier(nn.Module):
    def __init__(self, n_features, hidden=16):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return self.fc(h[-1]).squeeze(-1)


class CNNLSTMClassifier(nn.Module):
    def __init__(self, n_features, hidden=16, conv_channels=8):
        super().__init__()
        self.conv = nn.Conv1d(n_features, conv_channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.lstm = nn.LSTM(conv_channels, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, 1)

    def forward(self, x):
        xc = self.relu(self.conv(x.transpose(1, 2))).transpose(1, 2)
        _, (h, _) = self.lstm(xc)
        return self.fc(h[-1]).squeeze(-1)


def train_torch_classifier(model_cls, Xtr, ytr, n_features, epochs=15, lr=0.01, batch_size=64, seed=42):
    torch.manual_seed(seed)
    model = model_cls(n_features=n_features)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    Xtr_t = torch.tensor(Xtr, dtype=torch.float32)
    ytr_t = torch.tensor(ytr, dtype=torch.float32)
    n = len(Xtr_t)
    model.train()
    for epoch in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            opt.zero_grad()
            logits = model(Xtr_t[idx])
            loss = loss_fn(logits, ytr_t[idx])
            loss.backward()
            opt.step()
    model.eval()
    return model


def torch_predict_proba(model, X):
    with torch.no_grad():
        logits = model(torch.tensor(X, dtype=torch.float32))
        return torch.sigmoid(logits).numpy()


print("Model classes defined")


Model classes defined


## 5. Walk-Forward Evaluation (All 4 Models x 2 Horizons, 8 Chronological Folds Each)

Same `TimeSeriesSplit`, expanding-window pattern as the original, refactored into a
`run_walk_forward(target_col, horizon_days)` function so it can run once per horizon without
duplicating the loop. The Always-up baseline uses only information available before the test
row's date, same as every model here.

**Purge:** a training row's label looks `horizon_days` ahead, so any training row within
`horizon_days` of the test split has a label that reaches into the test period -- e.g. at the
30-day horizon, a training row 10 days before the split has a label built from a date 20 days
*into* the test set. That row's label already encodes test-period price movement before the
model ever sees a test row, which is a real leak the 1-day notebook never had (its labels
only reach 1 day past the split). Every fold now drops those training rows before fitting:
`train_idx` is filtered to exclude any index whose label window (`idx + horizon_days`) would
land at or past the test set's first index. At `horizon_days=7` this drops up to 6 rows per
fold; at 30 it drops up to 29 -- visible in the fold summaries below as `train` shrinking
relative to the pre-purge counts in the first run of this notebook.

In [6]:
def run_walk_forward(target_col, horizon_days, n_splits=8):
    X_all = model_frame[FEATURE_COLS].to_numpy()
    y_all = model_frame[target_col].to_numpy()
    dates_all = model_frame["date"].to_numpy()

    tscv = TimeSeriesSplit(n_splits=n_splits)
    results_rows = []
    fold_num = 0

    for train_idx_raw, test_idx in tscv.split(X_all):
        fold_num += 1

        # Purge: drop training rows whose label (horizon_days ahead) reaches into the test
        # period -- otherwise those rows' labels already encode test-period price movement.
        purge_mask = (train_idx_raw + horizon_days) >= test_idx.min()
        train_idx = train_idx_raw[~purge_mask]
        n_purged = len(train_idx_raw) - len(train_idx)

        Xtr_raw, ytr = X_all[train_idx], y_all[train_idx]
        Xte_raw, yte = X_all[test_idx], y_all[test_idx]
        dates_te = dates_all[test_idx]

        scaler = StandardScaler().fit(Xtr_raw)
        Xtr = scaler.transform(Xtr_raw)
        Xte = scaler.transform(Xte_raw)

        # --- Logistic Regression ---
        logit = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
        proba_logit = logit.predict_proba(Xte)[:, 1]

        # --- Random Forest ---
        rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, n_jobs=-1).fit(Xtr, ytr)
        proba_rf = rf.predict_proba(Xte)[:, 1]

        # --- LSTM / CNN-LSTM: sequence data spans the train/test boundary using only
        # already-known (training-fold) history for the earliest test-fold sequences.
        X_fold_full = np.concatenate([Xtr, Xte])
        y_fold_full = np.concatenate([ytr, yte])
        Xseq, yseq, _ = make_sequences(X_fold_full, y_fold_full, np.arange(len(X_fold_full)), SEQ_LOOKBACK)
        seq_target_idx = np.arange(SEQ_LOOKBACK, len(X_fold_full))
        test_seq_mask = seq_target_idx >= len(Xtr)
        Xseq_tr, yseq_tr = Xseq[~test_seq_mask], yseq[~test_seq_mask]
        Xseq_te, yseq_te = Xseq[test_seq_mask], yseq[test_seq_mask]

        lstm_model = train_torch_classifier(LSTMClassifier, Xseq_tr, yseq_tr, n_features=Xtr.shape[1])
        proba_lstm = torch_predict_proba(lstm_model, Xseq_te)

        cnn_lstm_model = train_torch_classifier(CNNLSTMClassifier, Xseq_tr, yseq_tr, n_features=Xtr.shape[1])
        proba_cnn_lstm = torch_predict_proba(cnn_lstm_model, Xseq_te)

        # --- Baseline ---
        always_up = np.ones(len(yte), dtype=int)

        for i in range(len(yte)):
            results_rows.append({
                "date": pd.Timestamp(dates_te[i]), "fold": fold_num, "actual": int(yte[i]),
                "proba_logit": proba_logit[i], "proba_rf": proba_rf[i],
                "proba_lstm": proba_lstm[i], "proba_cnn_lstm": proba_cnn_lstm[i],
                "always_up_pred": always_up[i],
            })
        print(f"  Fold {fold_num}: train={len(ytr)} (purged {n_purged})  test={len(yte)}  "
              f"test_seq={len(yseq_te)}  "
              f"({pd.Timestamp(dates_te[0]).date()} to {pd.Timestamp(dates_te[-1]).date()})")

    daily = pd.DataFrame(results_rows)
    for m in ["logit", "rf", "lstm", "cnn_lstm"]:
        daily[f"pred_{m}"] = (daily[f"proba_{m}"] >= 0.5).astype(int)
    return daily


daily_by_horizon = {}
for h, col in HORIZON_TARGETS.items():
    print(f"=== {h}-day horizon ({col}) ===")
    daily_by_horizon[h] = run_walk_forward(col, horizon_days=h)
    daily_by_horizon[h].to_csv(MK4_DIR / f"multiday_replication_daily_predictions_{h}d.csv", index=False)
    print(f"  Total pooled out-of-sample rows: {len(daily_by_horizon[h])}\n")


=== 7-day horizon (target_bullish_7d) ===


  Fold 1: train=172 (purged 7)  test=175  test_seq=175  (2019-12-17 to 2020-08-17)


  Fold 2: train=347 (purged 7)  test=175  test_seq=175  (2020-08-18 to 2021-04-19)


  Fold 3: train=522 (purged 7)  test=175  test_seq=175  (2021-04-20 to 2021-12-20)


  Fold 4: train=697 (purged 7)  test=175  test_seq=175  (2021-12-21 to 2022-08-22)


...

  Fold 4: train=674 (purged 30)  test=175  test_seq=175  (2021-12-21 to 2022-08-22)


  Fold 5: train=849 (purged 30)  test=175  test_seq=175  (2022-08-23 to 2023-04-24)


  Fold 6: train=1024 (purged 30)  test=175  test_seq=175  (2023-04-25 to 2023-12-25)


  Fold 7: train=1199 (purged 30)  test=175  test_seq=175  (2023-12-26 to 2024-08-26)


  Fold 8: train=1374 (purged 30)  test=175  test_seq=175  (2024-08-27 to 2025-04-28)
  Total pooled out-of-sample rows: 1400



## 6. Significance Testing

Same McNemar's test + fold-clustered bootstrap as the original, run once per horizon on that
horizon's own daily-predictions table -- the two horizons' summary tables are independent of
each other, stacked here only for side-by-side reading. As noted in the intro, the
within-fold overlap in longer-horizon targets makes the existing "8 clusters" caveat bite
harder than it does in the 1-day notebook.

In [7]:
def mcnemar(pred_col, actual_col, base_col, df):
    model_correct = (df[pred_col] == df[actual_col]).to_numpy()
    base_correct = (df[base_col] == df[actual_col]).to_numpy()
    b = int(np.sum(model_correct & ~base_correct))
    c = int(np.sum(~model_correct & base_correct))
    p = binomtest(b, b + c, 0.5).pvalue if (b + c) > 0 else float("nan")
    return b, c, p


def fold_cluster_bootstrap(pred_col, actual_col, base_col, df, n_boot=10000, seed=42):
    d = df.copy()
    d["model_correct"] = (d[pred_col] == d[actual_col]).astype(float)
    d["base_correct"] = (d[base_col] == d[actual_col]).astype(float)
    fold_stats = d.groupby("fold").agg(n=("model_correct", "size"),
                                        model_acc=("model_correct", "mean"),
                                        base_acc=("base_correct", "mean")).reset_index()
    n_folds = len(fold_stats)
    fold_ns = fold_stats["n"].to_numpy()
    fold_model_sum = (fold_stats["model_acc"] * fold_stats["n"]).to_numpy()
    fold_base_sum = (fold_stats["base_acc"] * fold_stats["n"]).to_numpy()
    rng = np.random.default_rng(seed)
    boot_edges = np.empty(n_boot)
    for i in range(n_boot):
        picks = rng.integers(0, n_folds, size=n_folds)
        total_n = fold_ns[picks].sum()
        boot_edges[i] = fold_model_sum[picks].sum() / total_n - fold_base_sum[picks].sum() / total_n
    lo, hi = np.percentile(boot_edges, [2.5, 97.5])
    observed = fold_model_sum.sum() / fold_ns.sum() - fold_base_sum.sum() / fold_ns.sum()
    return n_folds, observed, lo, hi


MODEL_NAMES = {"logit": "Logistic Regression", "rf": "Random Forest",
               "lstm": "LSTM", "cnn_lstm": "CNN-LSTM"}
BASELINE_NAMES = {"always_up_pred": "Always-up"}

summary_by_horizon = {}
for h, daily in daily_by_horizon.items():
    summary_rows = []
    for model_key, model_label in MODEL_NAMES.items():
        pred_col = f"pred_{model_key}"
        model_acc = (daily[pred_col] == daily["actual"]).mean()
        for base_col, base_label in BASELINE_NAMES.items():
            base_acc = (daily[base_col] == daily["actual"]).mean()
            b, c, p = mcnemar(pred_col, "actual", base_col, daily)
            n_folds, observed, lo, hi = fold_cluster_bootstrap(pred_col, "actual", base_col, daily)
            summary_rows.append({
                "horizon_days": h, "model": model_label, "baseline": base_label,
                "model_acc": model_acc, "baseline_acc": base_acc,
                "edge": observed, "mcnemar_p": p,
                "bootstrap_ci_lo": lo, "bootstrap_ci_hi": hi,
                "ci_excludes_zero": not (lo <= 0 <= hi),
            })
    summary_by_horizon[h] = pd.DataFrame(summary_rows)
    summary_by_horizon[h].to_csv(MK4_DIR / f"multiday_replication_significance_summary_{h}d.csv", index=False)

pd.set_option("display.float_format", lambda x: f"{x:.4f}")
summary_all = pd.concat(summary_by_horizon.values(), ignore_index=True)
summary_all


,horizon_days,model,baseline,model_acc,baseline_acc,edge,mcnemar_p,bootstrap_ci_lo,bootstrap_ci_hi,ci_excludes_zero
0,7,Logistic Regression,Always-up,0.5000,0.5486,-0.0486,0.0220,-0.1207,0.0279,False
1,7,Random Forest,Always-up,0.5136,0.5486,-0.0350,0.0559,-0.1007,0.0307,False
2,7,LSTM,Always-up,0.4814,0.5486,-0.0671,0.0004,-0.1450,0.0114,False
3,7,CNN-LSTM,Always-up,0.4850,0.5486,-0.0636,0.0005,-0.1421,0.0079,False
4,30,Logistic Regression,Always-up,0.4521,0.5729,-0.1207,0.0000,-0.2707,0.0500,False
5,30,Random Forest,Always-up,0.5657,0.5729,-0.0071,0.6825,-0.1179,0.1200,False
6,30,LSTM,Always-up,0.4979,0.5729,-0.0750,0.0002,-0.2600,0.0929,False
7,30,CNN-LSTM,Always-up,0.4679,0.5729,-0.1050,0.0000,-0.2464,0.0586,False


## 7. Per-Model, Per-Baseline Readout

In [8]:
for h, summary in summary_by_horizon.items():
    print(f"=== {h}-day horizon ===")
    for _, row in summary.iterrows():
        sig = "SIGNIFICANT" if row["ci_excludes_zero"] else "not significant"
        print(f"{row['model']:22s} vs {row['baseline']:12s}  acc={row['model_acc']:.4f}  "
              f"base={row['baseline_acc']:.4f}  edge={row['edge']:+.4f}  p={row['mcnemar_p']:.4f}  "
              f"CI=[{row['bootstrap_ci_lo']:+.4f}, {row['bootstrap_ci_hi']:+.4f}]  ({sig})")
    print()


=== 7-day horizon ===
Logistic Regression    vs Always-up     acc=0.5000  base=0.5486  edge=-0.0486  p=0.0220  CI=[-0.1207, +0.0279]  (not significant)
Random Forest          vs Always-up     acc=0.5136  base=0.5486  edge=-0.0350  p=0.0559  CI=[-0.1007, +0.0307]  (not significant)
LSTM                   vs Always-up     acc=0.4814  base=0.5486  edge=-0.0671  p=0.0004  CI=[-0.1450, +0.0114]  (not significant)
CNN-LSTM               vs Always-up     acc=0.4850  base=0.5486  edge=-0.0636  p=0.0005  CI=[-0.1421, +0.0079]  (not significant)

=== 30-day horizon ===
Logistic Regression    vs Always-up     acc=0.4521  base=0.5729  edge=-0.1207  p=0.0000  CI=[-0.2707, +0.0500]  (not significant)
Random Forest          vs Always-up     acc=0.5657  base=0.5729  edge=-0.0071  p=0.6825  CI=[-0.1179, +0.1200]  (not significant)
LSTM                   vs Always-up     acc=0.4979  base=0.5729  edge=-0.0750  p=0.0002  CI=[-0.2600, +0.0929]  (not significant)
CNN-LSTM               vs Always-up     acc=

## 8. Summary

**What this changes from the original:** the target (7-day and 30-day instead of 1-day, §3),
and a purge (§5) that drops training rows whose forward-looking label reaches into the test
period -- up to 6 rows per fold at 7 days, up to 29 at 30 days. Everything else (14 features,
4 models, 8-fold walk-forward, significance methodology) is identical to
`original_model_replication.ipynb`. The Persistence baseline has been removed as a comparison
across this project's notebooks -- Always-up is now the only baseline.

**With Persistence removed, nothing in this notebook is significant.** Every one of the 8
model/horizon combinations is statistically indistinguishable from Always-up -- most sit
below it numerically (7 of 8 have a negative point-estimate edge), none reach significance
in either direction. This notebook previously reported one standout result -- 30-day Random
Forest beating Persistence by +9.93pp, the only significant finding across the multi-day
experiments -- but that result lived entirely in the Persistence comparison and disappears
along with it. Against Always-up, that same 30-day Random Forest result was already not
significant (edge -0.71pp, unchanged by this edit).

**The purge is still doing real work independent of this change.** Before purging, 7-day
CNN-LSTM looked significantly worse than Always-up -- an artifact of training on rows whose
labels already encoded test-period price moves. That false signal is still gone after
purging; removing Persistence doesn't affect that part of the story.

**Bottom line:** at both 7-day and 30-day horizons, no model beats simply predicting
"bullish" every day, and none of this notebook's results are statistically defensible
against that baseline. Always-up itself is a strong bar here (54.9% at 7 days, 57.3% at 30
days, both higher than the 1-day baseline's 51.5%, since BTC trended up over most of this
window) -- and nothing tested clears it.